# Week 05 · Day 02 — Pandas Foundations
## Cleaning a Real (Messy) Dataset — Data Cleaning

**Goal of this notebook:** build a small, deliberately messy "sales" dataset, diagnose
exactly what is wrong with it, fix it on purpose (with a reason written down for every
fix), and answer real business questions with `groupby`.

Every step follows the same pattern: **question → check → interpretation**. Nothing gets
changed silently.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 100)
print("pandas version:", pd.__version__)

pandas version: 3.0.2


## 1. Build a messy dataset — on purpose

Before fixing anything, we need a dataset whose problems we know in advance. That's the
only way to *prove* a fix actually worked later, instead of just hoping it did.

The table below (`date`, `category`, `amount`, `customer_age`) has three deliberate
issues baked in:

1. A few missing values, scattered across **different** columns (not all in one place).
2. One category written two inconsistent ways: `"Electronics"` vs `"electronics"`.
3. Nothing else is wrong — so if `.info()` finds anything else later, that itself is a
   discovery worth noting.

In [2]:
data = {
    "date": pd.to_datetime([
        "2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05",
        "2024-01-06", "2024-01-07", "2024-01-08", "2024-01-09", "2024-01-10",
    ]),
    "category": [
        "Electronics", "electronics", "Clothing", "Clothing", "Electronics",
        "Groceries", "Groceries", "Clothing", "electronics", "Groceries",
    ],
    "amount": [250.0, 89.5, np.nan, 45.0, 310.0, 12.75, np.nan, 60.0, 199.99, 15.5],
    "customer_age": [34, 28, 45, np.nan, 52, 23, 31, np.nan, 40, 26],
}
data_v2 = {
    "patient_id": [201, 202, 203, 204, 205, 206, 207, 208, 209, 210],
    "arrival_time": [
        "08:15", "08:30", "08:45", "09:10", "09:25", 
        "09:40", "10:05", "10:15", "10:30", "11:00"
    ],
    "primary_complaint": [
        "Cardiology", "cardiology", "Trauma", "CARDIOLOGY", "Neurology", 
        "trauma", "Neurology", "Cardiology", "trauma", "Orthopedics"
    ],
    "triage_score": [1, 3, np.nan, 2, 4, 1, 3, np.nan, 2, 5], 
    "wait_time_min": [12.0, 45.0, 8.0, 22.0, 85.0, np.nan, 60.0, 15.0, 30.0, 110.0],
    "heart_rate_bpm": [115, np.nan, 140, 98, 82, 130, 76, 105, 88, 72]
}

sales = pd.DataFrame(data)
sales


,date,category,amount,customer_age
0,2024-01-01,Electronics,250.00,34.0
1,2024-01-02,electronics,89.50,28.0
2,2024-01-03,Clothing,NaN,45.0
3,2024-01-04,Clothing,45.00,NaN
4,2024-01-05,Electronics,310.00,52.0
5,2024-01-06,Groceries,12.75,23.0
6,2024-01-07,Groceries,NaN,31.0
7,2024-01-08,Clothing,60.00,NaN
8,2024-01-09,electronics,199.99,40.0
9,2024-01-10,Groceries,15.50,26.0


In [3]:
medical = pd.DataFrame(data_v2)
medical

,patient_id,arrival_time,primary_complaint,triage_score,wait_time_min,heart_rate_bpm
0,201,08:15,Cardiology,1.0,12.0,115.0
1,202,08:30,cardiology,3.0,45.0,NaN
2,203,08:45,Trauma,NaN,8.0,140.0
3,204,09:10,CARDIOLOGY,2.0,22.0,98.0
4,205,09:25,Neurology,4.0,85.0,82.0
5,206,09:40,trauma,1.0,NaN,130.0
6,207,10:05,Neurology,3.0,60.0,76.0
7,208,10:15,Cardiology,NaN,15.0,105.0
8,209,10:30,trauma,2.0,30.0,88.0
9,210,11:00,Orthopedics,5.0,110.0,72.0


## 2. Diagnose before touching anything

Four checks, always in this order, before writing a single line that *changes* data:

- `.head()` — quick visual look
- `.info()` — dtype + non-null count per column (fastest way to spot missing data)
- `.describe()` — summary statistics per numeric column
- `.isna().sum()` — exact missing-value count per column

In [4]:
sales.head()

,date,category,amount,customer_age
0,2024-01-01,Electronics,250.0,34.0
1,2024-01-02,electronics,89.5,28.0
2,2024-01-03,Clothing,NaN,45.0
3,2024-01-04,Clothing,45.0,NaN
4,2024-01-05,Electronics,310.0,52.0


In [5]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          10 non-null     datetime64[us]
 1   category      10 non-null     str           
 2   amount        8 non-null      float64       
 3   customer_age  8 non-null      float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 452.0 bytes


In [6]:
sales.describe()

,date,amount,customer_age
count,10,8.000000,8.000000
mean,2024-01-05 12:00:00,122.842500,34.875000
min,2024-01-01 00:00:00,12.750000,23.000000
25%,2024-01-03 06:00:00,37.625000,27.500000
50%,2024-01-05 12:00:00,74.750000,32.500000
75%,2024-01-07 18:00:00,212.492500,41.250000
max,2024-01-10 00:00:00,310.000000,52.000000
std,NaN,114.587349,10.034761


In [7]:
sales.isna().sum()

date            0
category        0
amount          2
customer_age    2
dtype: int64

**Findings (written down before any fix):**

- `amount` has **2** missing values (rows 2 and 6).
- `customer_age` has **2** missing values (rows 3 and 7).
- `category` has **no** missing values, but it has a *consistency* problem, not a
  *missingness* problem: `"Electronics"` and `"electronics"` are being counted as two
  different categories purely because of casing. `.isna()` can't catch this — it's not
  a NaN, it's a labeling mistake — so it has to be caught separately with
  `.value_counts()` in step 6.
- `date` and every other column are complete and correctly typed.

## 3. Decide a missing-data strategy — per column, not in bulk

"Just drop every row with a gap" is not one honest answer for the whole table — each
column's missing values mean something different, so each gets its own decision:

- **`amount` → `fillna()` with the column median.** Sales amount is the thing we're
  about to aggregate in `groupby` later; dropping rows would quietly shrink the
  category totals. The median is a defensible "typical sale" placeholder and is less
  skewed by the one large \$310 sale than the mean would be.
- **`customer_age` → `fillna()` with the column mean, rounded to a whole year.** Age
  isn't used in this notebook's final aggregation, so preserving the row (with a
  reasonable estimate) is more valuable than losing the row's `amount` and `category`
  data along with it.

Both are stated here explicitly rather than buried silently in code, per the "every fill
value is an assumption" rule.

In [8]:
amount_fill_value = sales["amount"].median()
age_fill_value = round(sales["customer_age"].mean())

sales["amount"] = sales["amount"].fillna(amount_fill_value)
sales["customer_age"] = sales["customer_age"].fillna(age_fill_value)

print(f"Filled amount NaNs with median = {amount_fill_value}")
print(f"Filled customer_age NaNs with mean (rounded) = {age_fill_value}")

sales.isna().sum()  # should be all zeros now

Filled amount NaNs with median = 74.75
Filled customer_age NaNs with mean (rounded) = 35


date            0
category        0
amount          0
customer_age    0
dtype: int64

> Note: we used `fillna()` for **both** columns here on purpose, so the notebook
> also has a case where `dropna()` would have been the *wrong* call to compare against
> — dropping rows 2, 3, 6, 7 would have thrown away 40% of a 10-row dataset over two
> single missing values each, which is a bad trade for a dataset this small.

## 4. `.loc` vs `.iloc` — proven, not assumed

- `.loc[]` selects by **label** (the index value / column name).
- `.iloc[]` selects by **integer position** (0, 1, 2, ... regardless of label).

On a freshly built DataFrame with the default `0, 1, 2, ...` index these look identical
— that's exactly what makes the mix-up easy to miss. We'll prove they match first, then
break that assumption by sorting.

In [9]:
# On the ORIGINAL (unsorted) DataFrame, index labels == integer positions,
# so .loc[2] and .iloc[2] point at the same row.
same_by_label = sales.loc[2]
same_by_position = sales.iloc[2]

print("Are they the same row (original order)?", same_by_label.equals(same_by_position))
same_by_label

Are they the same row (original order)? True


date            2024-01-03 00:00:00
category                   Clothing
amount                        74.75
customer_age                   45.0
Name: 2, dtype: object

In [10]:
# Now sort by amount. The row that USED TO be at position 2 keeps its original
# label (its index doesn't change), but a different row now sits at position 2.
sales_sorted = sales.sort_values("amount").reset_index(drop=False)  # keep old index visible as "index"
sales_sorted = sales_sorted.set_index("index")  # restore original labels as the index, but rows are now reordered

different_by_label = sales_sorted.loc[2]
different_by_position = sales_sorted.iloc[2]

print("Are they the same row AFTER sorting?", different_by_label.equals(different_by_position))
print("\n.loc[2]  -> label 2, wherever it now sits:")
print(different_by_label)
print("\n.iloc[2] -> whatever row is now physically 3rd (position index 2):")
print(different_by_position)

Are they the same row AFTER sorting? False

.loc[2]  -> label 2, wherever it now sits:
date            2024-01-03 00:00:00
category                   Clothing
amount                        74.75
customer_age                   45.0
Name: 2, dtype: object

.iloc[2] -> whatever row is now physically 3rd (position index 2):
date            2024-01-04 00:00:00
category                   Clothing
amount                         45.0
customer_age                   35.0
Name: 3, dtype: object


**Why they diverge:** sorting reorders the *rows* but does not renumber the
*index labels* — row that was originally labeled `2` keeps the label `2` no matter
where it ends up sitting. `.loc[2]` always chases the label `2`; `.iloc[2]` always grabs
whatever is physically in the 3rd slot (position 2). After a sort, those are no longer
guaranteed to be the same row — and the wrong one produces no error, just a silently
wrong answer.

## 5. Filter with `&` / `|`, then break `and`/`or` on purpose

Boolean filtering on a DataFrame is exactly yesterday's NumPy masking: build a
True/False Series from a condition, then use it to index the table.

In [11]:
# Correct: & with parentheses around EACH condition (operator precedence requires this)
high_value_electronics = sales[(sales["amount"] > 100) & (sales["category"].str.lower() == "electronics")]
high_value_electronics

,date,category,amount,customer_age
0,2024-01-01,Electronics,250.00,34.0
4,2024-01-05,Electronics,310.00,52.0
8,2024-01-09,electronics,199.99,40.0


In [12]:
# Now deliberately reproduce the classic mistake: plain Python `and`/`or`
# instead of the vectorized `&`/`|`.
try:
    broken = sales[(sales["amount"] > 100) and (sales["category"].str.lower() == "electronics")]
except ValueError as e:
    print("It broke, exactly as expected:")
    print(f"  ValueError: {e}")

It broke, exactly as expected:
  ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


**Why it breaks:** Python's `and`/`or` are built for a single `True`/`False` value
— they can't evaluate "truthiness" of an entire Series of many values at once, so pandas
raises `ValueError: The truth value of a Series is ambiguous`. `&`/`|` are the
vectorized, element-by-element versions built for exactly this. The parentheses around
each condition are required too, because `>` and `==` have lower precedence than `&`,
so without them Python tries to apply `&` before the comparisons even run.

## 6. Find and fix the inconsistent category

`.value_counts()` is often the fastest way to *discover* this kind of problem — a column
that looks clean until you count it.

In [13]:
print("Before fix:")
print(sales["category"].value_counts())

# One vectorized string operation across the WHOLE column — never hand-edit individual rows.
sales["category"] = sales["category"].str.lower()

print("\nAfter fix:")
print(sales["category"].value_counts())

Before fix:
category
Clothing       3
Groceries      3
Electronics    2
electronics    2
Name: count, dtype: int64

After fix:
category
electronics    4
clothing       3
groceries      3
Name: count, dtype: int64


## 7. `groupby` — split, apply, combine

Now that `category` is clean, we can trust a per-category aggregation. `groupby` splits
the rows into groups by category, computes an aggregate independently inside each group,
and combines the results — turning a manual filter-and-loop into one line.

In [14]:
category_stats = sales.groupby("category")["amount"].agg(mean_amount="mean", transaction_count="count")
category_stats = category_stats.sort_values("mean_amount", ascending=False)
category_stats

,mean_amount,transaction_count
category,,
electronics,212.372500,4
clothing,59.916667,3
groceries,34.333333,3


In [15]:
total_by_category = sales.groupby("category")["amount"].sum().sort_values(ascending=False)
top_category = total_by_category.index[0]

print("Total amount by category (highest first):")
print(total_by_category)
print(f"\nTop category by total amount: {top_category!r} (${total_by_category.iloc[0]:.2f})")

Total amount by category (highest first):
category
electronics    849.49
clothing       179.75
groceries      103.00
Name: amount, dtype: float64

Top category by total amount: 'electronics' ($849.49)


## 8. Vectorized operation vs. `.apply()` — timed, not assumed

Prefer a direct vectorized expression over `.apply()` with a Python function whenever
the operation is plain arithmetic — a vectorized op runs as a compiled loop, while
`.apply()` calls a Python function once per row. We'll prove the gap on a large
synthetic version of the table (built by repeating our 10 cleaned rows) rather than just
asserting it.

In [16]:
REPEATS = 30_000  # 10 rows * 30,000 = 300,000 rows
big_sales = pd.concat([sales] * REPEATS, ignore_index=True)
big_sales["quantity"] = np.random.randint(1, 10, size=len(big_sales))
print(f"big_sales has {len(big_sales):,} rows")
big_sales.head()

big_sales has 300,000 rows


,date,category,amount,customer_age,quantity
0,2024-01-01,electronics,250.00,34.0,2
1,2024-01-02,electronics,89.50,28.0,1
2,2024-01-03,clothing,74.75,45.0,7
3,2024-01-04,clothing,45.00,35.0,9
4,2024-01-05,electronics,310.00,52.0,9


In [17]:
%%timeit
big_sales["total_vectorized"] = big_sales["amount"] * big_sales["quantity"]

703 μs ± 31.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [18]:
%%timeit
big_sales["total_apply"] = big_sales.apply(lambda row: row["amount"] * row["quantity"], axis=1)

1.96 s ± 93.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
# Confirm both approaches actually produce the same numbers, not just compare their speed.
big_sales["total_vectorized"] = big_sales["amount"] * big_sales["quantity"]
big_sales["total_apply"] = big_sales.apply(lambda row: row["amount"] * row["quantity"], axis=1)

print("Results identical:", big_sales["total_vectorized"].equals(big_sales["total_apply"]))
big_sales[["amount", "quantity", "total_vectorized", "total_apply"]].head()

Results identical: True


,amount,quantity,total_vectorized,total_apply
0,250.00,2,500.00,500.00
1,89.50,1,89.50,89.50
2,74.75,7,523.25,523.25
3,45.00,9,405.00,405.00
4,310.00,9,2790.00,2790.00


**Reading the `%%timeit` numbers above:** the vectorized version runs the
multiplication once, across the whole column, as compiled (NumPy-backed) code. The
`.apply()` version calls a Python lambda function **300,000 separate times** — once per
row — paying Python's per-call overhead every single time. In this run the vectorized
version finished in well under a millisecond, while `.apply()` took over a second —
roughly **1,000–3,000x slower**, and the gap only grows on bigger tables. `.apply()`
still earns its place for logic that genuinely can't be written as plain arithmetic —
but it's a deliberate choice, never the default reach.

## 9. Summary


| Step | Question | Check | Finding / Fix |
|---|---|---|---|
| Diagnose | Is anything missing or wrong? | `.info()`, `.isna().sum()`, `.value_counts()` | 2 NaNs in `amount`, 2 in `customer_age`, `category` case mismatch |
| Missing data | Fill or drop? | Per column | `amount` → median, `customer_age` → mean (both filled, not dropped) |
| `.loc` vs `.iloc` | Do they agree? | Compared before/after sort | Match before sorting, diverge after — label ≠ position once order changes |
| Filtering | `&` vs `and` | Ran both on purpose | `&` works element-wise; `and` raises `ValueError` |
| Category mess | Real duplicate categories? | `.value_counts()` | `"Electronics"`/`"electronics"` → merged with `.str.lower()` |
| `groupby` | Best category by revenue? | `groupby("category")` | Highest total-amount category identified above |
| Vectorized vs `.apply()` | How much slower is `.apply()`? | `%%timeit` on 300k rows | Vectorized wins by roughly an order of magnitude |



In [20]:
medical.head()

,patient_id,arrival_time,primary_complaint,triage_score,wait_time_min,heart_rate_bpm
0,201,08:15,Cardiology,1.0,12.0,115.0
1,202,08:30,cardiology,3.0,45.0,NaN
2,203,08:45,Trauma,NaN,8.0,140.0
3,204,09:10,CARDIOLOGY,2.0,22.0,98.0
4,205,09:25,Neurology,4.0,85.0,82.0


In [21]:
medical.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patient_id         10 non-null     int64  
 1   arrival_time       10 non-null     str    
 2   primary_complaint  10 non-null     str    
 3   triage_score       8 non-null      float64
 4   wait_time_min      9 non-null      float64
 5   heart_rate_bpm     9 non-null      float64
dtypes: float64(3), int64(1), str(2)
memory usage: 612.0 bytes


In [22]:
medical.describe()

,patient_id,triage_score,wait_time_min,heart_rate_bpm
count,10.00000,8.000000,9.00000,9.000000
mean,205.50000,2.625000,43.00000,100.666667
std,3.02765,1.407886,35.57738,23.921747
min,201.00000,1.000000,8.00000,72.000000
25%,203.25000,1.750000,15.00000,82.000000
50%,205.50000,2.500000,30.00000,98.000000
75%,207.75000,3.250000,60.00000,115.000000
max,210.00000,5.000000,110.00000,140.000000


In [23]:
medical.isna().sum()

patient_id           0
arrival_time         0
primary_complaint    0
triage_score         2
wait_time_min        1
heart_rate_bpm       1
dtype: int64

In [24]:
triage_fill = np.floor(medical["triage_score"].median())
wait_time_fill = np.floor(medical["wait_time_min"].median())
heart_rate_fill = np.floor(medical["heart_rate_bpm"].median())

medical["triage_score"] = medical["triage_score"].fillna(triage_fill)
medical["wait_time_min"] = medical["wait_time_min"].fillna(wait_time_fill)
medical["heart_rate_bpm"] = medical["heart_rate_bpm"].fillna(heart_rate_fill)
# print(heart_rate_fill)
medical

,patient_id,arrival_time,primary_complaint,triage_score,wait_time_min,heart_rate_bpm
0,201,08:15,Cardiology,1.0,12.0,115.0
1,202,08:30,cardiology,3.0,45.0,98.0
2,203,08:45,Trauma,2.0,8.0,140.0
3,204,09:10,CARDIOLOGY,2.0,22.0,98.0
4,205,09:25,Neurology,4.0,85.0,82.0
5,206,09:40,trauma,1.0,30.0,130.0
6,207,10:05,Neurology,3.0,60.0,76.0
7,208,10:15,Cardiology,2.0,15.0,105.0
8,209,10:30,trauma,2.0,30.0,88.0
9,210,11:00,Orthopedics,5.0,110.0,72.0


In [25]:
medical["primary_complaint"].value_counts()

primary_complaint
Cardiology     2
Neurology      2
trauma         2
cardiology     1
Trauma         1
CARDIOLOGY     1
Orthopedics    1
Name: count, dtype: int64

In [26]:
medical["primary_complaint"] = (medical["primary_complaint"].str.strip().str.title())
medical["primary_complaint"].value_counts()

primary_complaint
Cardiology     4
Trauma         3
Neurology      2
Orthopedics    1
Name: count, dtype: int64

In [27]:
print("Filled Table")
medical

Filled Table


,patient_id,arrival_time,primary_complaint,triage_score,wait_time_min,heart_rate_bpm
0,201,08:15,Cardiology,1.0,12.0,115.0
1,202,08:30,Cardiology,3.0,45.0,98.0
2,203,08:45,Trauma,2.0,8.0,140.0
3,204,09:10,Cardiology,2.0,22.0,98.0
4,205,09:25,Neurology,4.0,85.0,82.0
5,206,09:40,Trauma,1.0,30.0,130.0
6,207,10:05,Neurology,3.0,60.0,76.0
7,208,10:15,Cardiology,2.0,15.0,105.0
8,209,10:30,Trauma,2.0,30.0,88.0
9,210,11:00,Orthopedics,5.0,110.0,72.0


In [28]:
# Now no missing values
medical.isna().sum()

patient_id           0
arrival_time         0
primary_complaint    0
triage_score         0
wait_time_min        0
heart_rate_bpm       0
dtype: int64

In [29]:
complaints_stat = medical.groupby("primary_complaint").agg(patient_count = ("patient_id","count"),
                                                           most_common_triangle_score = ("triage_score",lambda x: x.mode().iloc[0])).reset_index()
complaints_stat

,primary_complaint,patient_count,most_common_triangle_score
0,Cardiology,4,2.0
1,Neurology,2,3.0
2,Orthopedics,1,5.0
3,Trauma,3,2.0


In [30]:
medical.groupby(["primary_complaint", "triage_score"]).size().reset_index(name="patient_count")

,primary_complaint,triage_score,patient_count
0,Cardiology,1.0,1
1,Cardiology,2.0,2
2,Cardiology,3.0,1
3,Neurology,3.0,1
4,Neurology,4.0,1
5,Orthopedics,5.0,1
6,Trauma,1.0,1
7,Trauma,2.0,2
